# YOLOv8 Detection of Muzzle Regions: Left Nostril, Right Nostril, Upper Lip, Lower Lip

## 1. Overview

This notebook implements an object detection workflow based on YOLOv8 (Ultralytics). The task is to detect four labeled regions in animal muzzle images: left nostril, right nostril, upper lip, and lower lip. The source annotations are CVAT-style bounding boxes exported as YOLO text files.

The workflow covers dataset preparation, annotation processing, class mapping, a deterministic train/validation split grouped by source video, YOLO dataset construction, model training, validation, and inference. Training was performed with GPU acceleration in Google Colab.

The dataset, annotations, and trained weights are not distributed with this notebook. Dataset sizes are withheld and appear as `X`. The model metrics in the Results section are the values recorded from the training run described below.

## 2. Environment and Configuration

The notebook uses Ultralytics 8.4.138 and PyTorch 2.11.0. PyTorch is provided by the Colab runtime, and Ultralytics is pinned to the version used for the recorded run.

In [ ]:
%pip install -q ultralytics==8.4.138

All paths and training settings are defined in one cell. The paths are neutral defaults and should be adjusted to the user's environment. `RUNS_DIR` is on the local Colab filesystem by default and should point to persistent storage if the weights need to outlive the session.

`RANDOM_SEED` and `SPLIT_RATIO` determine the train/validation split and should be kept fixed to reproduce the same split.

In [ ]:
import random
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import NamedTuple

import pandas as pd
import torch
from IPython.display import Image, display
from ultralytics import YOLO

DATASET_ARCHIVE_PATH = Path("/content/input/dataset_archive.zip")
EXTRACT_DIR = Path("/content/raw_data")
PREPARED_DATASET_DIR = Path("/content/dataset")
RUNS_DIR = Path("/content/training_runs")
RUN_NAME = "yolov8s_region_detection"

SPLIT_RATIO = 0.85
RANDOM_SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
SOURCE_TO_TARGET_CLASS = {0: 0, 1: 1, 3: 2, 4: 3}
CLASS_NAMES = {0: "left nostril", 1: "right nostril", 2: "upper lip", 3: "lower lip"}

MODEL_CHECKPOINT = "yolov8s.pt"
EPOCHS = 150
IMG_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 30
SAVE_PERIOD = -1
OPTIMIZER = "AdamW"
FLIPLR = 0.5

Training requires a CUDA-capable GPU. The cell below reports the PyTorch version and whether CUDA is available.

In [3]:
cuda_available = torch.cuda.is_available()
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {cuda_available}")

DEVICE = 0 if cuda_available else "cpu"
if not cuda_available:
    print("No CUDA device detected. Enable a GPU runtime before training.")

PyTorch version: 2.11.0+cu128
CUDA available:  True


## 3. Dataset Preparation

The dataset is expected as a zip archive containing two directories, at any nesting depth:

- `images`: frame images with extensions `.jpg`, `.jpeg`, or `.png`
- `boxes`: one YOLO-format `.txt` label file per image, sharing the image's filename stem. Each line holds a class index followed by the normalized bounding-box coordinates.

The images are frames extracted from short videos. The archive is extracted and the `images` and `boxes` directories are located by name.

In [ ]:
def extract_archive(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)


def find_directory(root, name):
    matches = sorted(path for path in root.rglob(name) if path.is_dir())
    if not matches:
        raise FileNotFoundError(f"No directory named '{name}' found under {root}")
    return matches[0]


extract_archive(DATASET_ARCHIVE_PATH, EXTRACT_DIR)
LABELS_DIR = find_directory(EXTRACT_DIR, "boxes")
IMAGES_DIR = find_directory(EXTRACT_DIR, "images")

## 4. Annotation Processing

Images and labels are paired by filename stem. Label files without a corresponding image are excluded, and images without a label file are excluded from the prepared dataset. Only matched image-label pairs are used in the following steps.

In [ ]:
class SamplePair(NamedTuple):
    stem: str
    image_name: str
    label_name: str


image_names_by_stem = {}
for image_path in sorted(IMAGES_DIR.iterdir()):
    if image_path.suffix.lower() in IMAGE_EXTENSIONS:
        image_names_by_stem[image_path.stem] = image_path.name

label_names = sorted(path.name for path in LABELS_DIR.iterdir() if path.suffix == ".txt")

matched_pairs = []
for label_name in label_names:
    stem = Path(label_name).stem
    if stem in image_names_by_stem:
        matched_pairs.append(SamplePair(stem, image_names_by_stem[stem], label_name))

## 5. Class Mapping

The source annotations use five class indices, including a `muzzle` class (index 2) that is not part of this detection task. Label lines of that class are dropped, and the remaining classes are remapped to contiguous indices 0 to 3.

| Source index | Source class | Target index | Target class |
|---|---|---|---|
| 0 | left nostril | 0 | left nostril |
| 1 | right nostril | 1 | right nostril |
| 2 | muzzle | dropped | not used |
| 3 | upper lip | 2 | upper lip |
| 4 | lower lip | 3 | lower lip |

The remapping is applied while label files are written into the prepared dataset. Box coordinates are copied unchanged. In the recorded run, the Ultralytics logs reported no background images in either split, so no image lost all of its labels through this step.

In [ ]:
def remap_label_file(source_path, destination_path):
    """Copy a YOLO label file, dropping unmapped classes and remapping class indices."""
    with source_path.open() as source, destination_path.open("w") as destination:
        for line in source:
            fields = line.split()
            if not fields:
                continue
            source_class = int(fields[0])
            if source_class not in SOURCE_TO_TARGET_CLASS:
                continue
            target_class = SOURCE_TO_TARGET_CLASS[source_class]
            destination.write(" ".join([str(target_class)] + fields[1:]) + "\n")

## 6. Grouped Train/Validation Split

The images are frames extracted from short videos, so consecutive frames from the same video are nearly identical. A random frame-level split would place near-duplicate frames in both the training and validation sets. This leaks information between the splits and inflates validation metrics, because the model is evaluated on frames that closely resemble ones it was trained on.

To avoid this, frames are grouped by source video or sample before splitting. The group key is the filename stem with its trailing frame index removed (for example, `sample_0001_000000` and `sample_0001_000001` share the group `sample_0001`). Whole groups are assigned to either training or validation, so no source appears in both sets.

The split procedure is deterministic:

1. Group keys are sorted, then shuffled with `random.seed(RANDOM_SEED)`.
2. Groups are added to the training set in shuffled order until the number of training frames reaches `int(len(matched_pairs) * SPLIT_RATIO)`.
3. All remaining groups form the validation set.

Because whole groups are assigned together, the training set can slightly exceed the target frame count. The same seed always produces the same split. The cell reports the number of sources that appear in both splits, which is expected to be 0.

In [7]:
def get_group_key(stem):
    """Return the source video or sample identifier by removing the trailing frame index."""
    return stem.rsplit("_", 1)[0]


pairs_by_group = defaultdict(list)
for pair in matched_pairs:
    pairs_by_group[get_group_key(pair.stem)].append(pair)

group_keys = sorted(pairs_by_group)
random.seed(RANDOM_SEED)
random.shuffle(group_keys)

target_train_count = int(len(matched_pairs) * SPLIT_RATIO)
train_pairs, val_pairs = [], []
train_groups, val_groups = [], []
for group_key in group_keys:
    if len(train_pairs) < target_train_count:
        train_pairs.extend(pairs_by_group[group_key])
        train_groups.append(group_key)
    else:
        val_pairs.extend(pairs_by_group[group_key])
        val_groups.append(group_key)

shared_groups = set(train_groups) & set(val_groups)
print(f"Sources present in both splits: {len(shared_groups)}")

Sources present in both splits: 0


## 7. YOLO Dataset Construction

The prepared dataset directory is deleted and rebuilt on every run, so repeated execution cannot accumulate or duplicate files across splits. Images are copied into `images/train` and `images/val`, and the remapped label files are written into `labels/train` and `labels/val`. A `data.yaml` file describes the dataset for Ultralytics.

In [ ]:
def build_split(pairs, split_name):
    image_dir = PREPARED_DATASET_DIR / "images" / split_name
    label_dir = PREPARED_DATASET_DIR / "labels" / split_name
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    for pair in pairs:
        shutil.copy(IMAGES_DIR / pair.image_name, image_dir / pair.image_name)
        remap_label_file(LABELS_DIR / pair.label_name, label_dir / pair.label_name)


shutil.rmtree(PREPARED_DATASET_DIR, ignore_errors=True)
build_split(train_pairs, "train")
build_split(val_pairs, "val")

In [9]:
class_lines = "\n".join(f"  {index}: {name}" for index, name in CLASS_NAMES.items())
dataset_yaml = (
    f"path: {PREPARED_DATASET_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    f"{class_lines}\n"
)

dataset_yaml_path = PREPARED_DATASET_DIR / "data.yaml"
dataset_yaml_path.write_text(dataset_yaml)
print(dataset_yaml)

path: /content/dataset
train: images/train
val: images/val
names:
  0: left nostril
  1: right nostril
  2: upper lip
  3: lower lip



### Dataset statistics

Exact dataset sizes are withheld.

| Statistic | Value |
|---|---|
| Source videos or samples | X |
| Matched image-label pairs | X |
| Training images | X |
| Validation images | X |
| Validation instances | X |
| Classes | 4 |
| Training fraction target | 0.85 of matched pairs |

## 8. Model Training

The detector is fine-tuned from the pretrained `yolov8s.pt` checkpoint with the number of classes set to 4. Training uses AdamW, set explicitly so that the optimizer does not depend on the Ultralytics version. Early stopping is enabled with `patience=30`: training stops when validation fitness has not improved for 30 consecutive epochs. The checkpoint with the best validation fitness is saved as `best.pt`, and the most recent epoch is saved as `last.pt`.

Horizontal flip probability is set explicitly with `fliplr=0.5`, which equals the Ultralytics default. Other hyperparameters not set in the cell below were left at the Ultralytics 8.4.138 defaults, including mosaic augmentation (1.0), 10 final epochs without mosaic (`close_mosaic`), and 3 warmup epochs. The Ultralytics training seed was left at its default value of 0; `RANDOM_SEED` applies to the train/validation split only.

Horizontal flip augmentation was enabled (`fliplr=0.5`) during the recorded training run. No separate ablation study was performed to measure its effect on side-specific classes.

Per-epoch console output is omitted from this notebook, and the cell output below keeps only the lines that summarize the recorded run.

In [10]:
model = YOLO(MODEL_CHECKPOINT)
training_results = model.train(
    data=str(dataset_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    save_period=SAVE_PERIOD,
    optimizer=OPTIMIZER,
    fliplr=FLIPLR,
)

optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 46, best model saved as best.pt.


### Training summary

| Item | Value |
|---|---|
| Model | `yolov8s`, initialized from `yolov8s.pt` (349 of 355 items transferred from the pretrained weights) |
| Model size | 11,137,148 parameters, 130 layers, 28.6 GFLOPs (training graph) |
| Classes | 4 |
| Image size | 640 |
| Batch size | 16 |
| Maximum epochs | 150 |
| Early stopping patience | 30 epochs |
| Optimizer | AdamW (initial learning rate 0.01, momentum 0.937, weight decay 0.0005) |
| Mixed precision (AMP) | Enabled |
| Hardware | Single GPU, Google Colab |
| Horizontal flip probability (`fliplr`) | 0.5 |
| Epochs completed | 76 |
| Best epoch | 46 |

Training did not run for all 150 configured epochs. Early stopping ended the run after 76 epochs, 30 epochs after the best result at epoch 46.

## 9. Model Validation

The model object returned by the training session holds the `best.pt` weights. It is validated on the validation split, which is the same split used for monitoring during training. The metrics are computed by Ultralytics at its default confidence and IoU thresholds for validation.

The raw metrics object is converted into a table of overall and per-class precision, recall, mAP50, and mAP50-95.

In [ ]:
validation_metrics = model.val(verbose=False)

In [12]:
def summarize_detection_metrics(metrics, class_names):
    """Return overall and per-class detection metrics as a DataFrame."""
    box = metrics.box
    rows = [
        {
            "Class": "all",
            "Precision": box.mp,
            "Recall": box.mr,
            "mAP50": box.map50,
            "mAP50-95": box.map,
        }
    ]
    for position, class_index in enumerate(box.ap_class_index):
        precision, recall, map50, map50_95 = box.class_result(position)
        rows.append(
            {
                "Class": class_names[int(class_index)],
                "Precision": precision,
                "Recall": recall,
                "mAP50": map50,
                "mAP50-95": map50_95,
            }
        )
    return pd.DataFrame(rows)


metrics_table = summarize_detection_metrics(validation_metrics, CLASS_NAMES)
print(metrics_table.to_string(index=False, float_format="{:.3f}".format))

        Class  Precision  Recall  mAP50  mAP50-95
          all      0.872   0.870  0.918     0.621
 left nostril      0.807   0.775  0.859     0.554
right nostril      0.778   0.785  0.846     0.546
    upper lip      0.968   0.975  0.988     0.700
    lower lip      0.935   0.943  0.981     0.682


## 10. Results

All values were measured in the recorded run and are reported as logged, rounded to three decimal places.

### Overall validation metrics, best checkpoint

| Metric | Value |
|---|---:|
| Precision | 0.872 |
| Recall | 0.870 |
| mAP50 | 0.918 |
| mAP50-95 | 0.621 |

### Per-class validation metrics, best checkpoint

| Class | Precision | Recall | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|
| left nostril | 0.807 | 0.775 | 0.859 | 0.554 |
| right nostril | 0.778 | 0.785 | 0.846 | 0.546 |
| upper lip | 0.968 | 0.975 | 0.988 | 0.700 |
| lower lip | 0.935 | 0.943 | 0.981 | 0.682 |

### Validation performed automatically at the end of training

Ultralytics also validated `best.pt` at the end of training. The values differ slightly from the tables above in the third decimal place. They are reported for completeness.

| Class | Precision | Recall | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|
| all | 0.869 | 0.869 | 0.916 | 0.620 |
| left nostril | 0.800 | 0.774 | 0.856 | 0.554 |
| right nostril | 0.774 | 0.784 | 0.844 | 0.545 |
| upper lip | 0.968 | 0.975 | 0.988 | 0.698 |
| lower lip | 0.934 | 0.945 | 0.976 | 0.681 |

### Scope of the evaluation

- The validation split is grouped by source video or sample, so no source appears in both training and validation.
- The same validation split was used for early stopping and for selecting `best.pt`. No separate held-out test set was evaluated, so the reported metrics describe this validation split only.
- Training curves, confusion matrices, and prediction examples were generated by Ultralytics in the run directory but are not displayed in this notebook.

## 11. Inference

The best checkpoint is loaded from the training run directory. `detect_regions` runs detection on one image at a configurable confidence threshold, saves the annotated image, and returns the Ultralytics result. `print_detections` lists the predicted class, confidence, and box coordinates in pixels.

In [ ]:
BEST_WEIGHTS = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
inference_model = YOLO(str(BEST_WEIGHTS))


def detect_regions(model, image_path, confidence_threshold=0.25, output_path="prediction_output.jpg"):
    """Run detection on one image, save the annotated image, and return the result."""
    result = model.predict(str(image_path), conf=confidence_threshold, device=DEVICE)[0]
    result.save(filename=str(output_path))
    return result


def print_detections(result):
    for box in result.boxes:
        class_name = result.names[int(box.cls)]
        x_min, y_min, x_max, y_max = (round(value) for value in box.xyxy[0].tolist())
        print(f"{class_name}: confidence {float(box.conf):.3f}, box ({x_min}, {y_min}, {x_max}, {y_max})")

Set `NEW_IMAGE_PATH` to an image of a muzzle and run the cell below. The annotated image is written to `OUTPUT_IMAGE_PATH` and displayed. No prediction output is included in this notebook.

In [ ]:
NEW_IMAGE_PATH = "/content/sample_image.jpg"
OUTPUT_IMAGE_PATH = "prediction_output.jpg"

if Path(NEW_IMAGE_PATH).exists():
    prediction = detect_regions(inference_model, NEW_IMAGE_PATH, output_path=OUTPUT_IMAGE_PATH)
    print_detections(prediction)
    display(Image(filename=OUTPUT_IMAGE_PATH))
else:
    print(f"Image not found: {NEW_IMAGE_PATH}. Set NEW_IMAGE_PATH to an existing image and run this cell again.")